# Inspect the results of the assessment of a classifier's fairness based on movement patterns, focusing on single-cell candidates

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import pickle

import folium
import branca.colormap as cm

In [ ]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)

# Read the results of an assessment of fairness based on movement patterns from disk.
path_results = './res_exp.pkl'
with open(path_results, "rb") as f:
    dict_res = pickle.load(f)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

For each candidate, retrieve the grid and subset of cell it refers to.

In [ ]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Compute the number of objects associated with each candidate
num_objs_candidates = np.diff(dict_candidates['start_pos'])

# Retrieve the log-likelihood ratios computed for the candidates.
extreme_lr_threshold = dict_res['threshold_extreme']
vec_lr_dataset = dict_res['vec_LR_dataset']
vec_inrate_dataset = dict_res['vec_inrate_dataset']
vec_outrate_dataset = dict_res['vec_outrate_dataset']
labels = dict_res['dataset']
# display(vec_lr_dataset)


# For each candidate, here represented as a tuple of cell IDs, associate the grid and subset of cells it refers to.
num_candidates = vec_lr_dataset.size
list_grid_ids = np.empty(num_candidates, dtype=object)
list_cellids = np.empty(num_candidates, dtype=object)
count = 0
for grid in grid_info:
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=object)
    grid_id[0] = (int(grid[1]), int(grid[2]))
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    list_grid_ids[count : count + num_els_grid] = grid_id
    list_cellids[count : count + num_els_grid] = cell_ids

    count += num_els_grid


# Put all the information in a pandas Dataframe.
df_candidates = pd.DataFrame({
    "grid_id":   list_grid_ids,
    "cell_ids":  list_cellids,
    "lr":        vec_lr_dataset.astype(np.float32),
    "num_objs":  num_objs_candidates.astype(np.uint32),
    "in_rate":   vec_inrate_dataset.astype(np.float32),
    "out_rate":  vec_outrate_dataset.astype(np.float32)
})
#
# Count the number of cells making up each candidate.
# NOTE: We use numpy's 'fromiter' because it's way faster (C-backed code) than using pandas' apply/map + lamba func on a series.
cell_ids = df_candidates['cell_ids'].to_numpy()
df_candidates['num_cells'] = np.fromiter(
    (len(x) if type(x) is tuple else 1 for x in cell_ids),
    dtype=np.uint32,
    count=cell_ids.size
)
del cell_ids
# print(df_candidates)


# Free some memory.
del list_grid_ids, list_cellids, vec_lr_dataset, num_objs_candidates, vec_inrate_dataset, vec_outrate_dataset
del dict_candidates, dict_res

In [ ]:
print(f"Extreme threshold log-lr: {extreme_lr_threshold}")
df_candidates_extreme = df_candidates.loc[df_candidates['lr'] > extreme_lr_threshold]
print(f"Number of extreme candidates: {len(df_candidates_extreme)}")
# print(df_candidates_extreme['num_cells'].value_counts())

# Select the candidates made up of more than a certain number of cells, and then sort them
# according to the compute log-lr.
# df_candidates_extreme = df_candidates_extreme.loc[df_candidates_extreme['num_cells'] >= 1]
df_candidates_extreme = df_candidates_extreme.sort_values('lr', ascending=False)
df_candidates_extreme


# For every candidate, retrieve the geopandas of the grid associated with the candidate; then, compute the candidate's polygon
# or multipolygon (in case the candidate is made of multiple cells).
list_poly_candidates = []
for grid_id, cell_ids in zip(df_candidates_extreme['grid_id'], df_candidates_extreme['cell_ids']) :
    list_poly_candidates.append(dict_grids[grid_id].grid.loc[np.atleast_1d(cell_ids)].union_all())
df_candidates_extreme['geometry'] = list_poly_candidates
crs = next(iter(dict_grids.values())).grid.crs

df_candidates_extreme = gpd.GeoDataFrame(df_candidates_extreme, crs=crs, geometry='geometry')
display(df_candidates_extreme)

#### Multi-cell candidate plot

In [ ]:
import folium
from folium.plugins import GroupedLayerControl

df = df_candidates_extreme.copy()

# Base map centered on candidates
minx, miny, maxx, maxy = df.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=12, prefer_canvas=True)

# Create groups of layers, with each group grouping candidates of the same size, and with one layer per candidate.
for size_cand in range(df['num_cells'].max()) :

    df_sel = df.loc[df['num_cells'] == size_cand]
    if len(df_sel) == 0 : continue

    candidate_layers = []
    for _, row in df_sel.iterrows():
        layer_name = f"G:{row['grid_id']}, CA:{row['cell_ids']}, LR:{row['lr']}"
        fg = folium.FeatureGroup(name=layer_name, show=False)  # hidden initially

        str_tooltip = f"Candidate ID: {row['cell_ids']}<br>" + \
                      f"Log-LR: {row['lr']:.3f}<br>" + \
                      f"Number objects: {row['num_objs']}<br>" + \
                      f"Positive inrate: {row['in_rate']:.3f}<br>" + \
                      f"Positive outrate: {row['out_rate']:.3f}"
        folium.GeoJson(row["geometry"].__geo_interface__, tooltip = str_tooltip).add_to(fg)
        fg.add_to(m)
        candidate_layers.append(fg)

    # Group the extreme candidates of the same size in the same group.
    GroupedLayerControl(groups={f"Extreme candidates of size {size_cand}": candidate_layers},
                        exclusive_groups=False,   # radio buttons (mutually exclusive)
                        collapsed=True,         # make the group compact
                    ).add_to(m)

m